# ETL Pipeline: Mobile Gaming Platform (Google Play Store)
**Mục tiêu:** Tiền xử lý, làm sạch và chuẩn hóa dữ liệu Mobile từ `data/raw/mobile_playstore_games_2000+.csv`.

### Các bước xử lý chính:
1. Đọc và xử lý trường `realInstalls` làm chỉ số Volume.
2. Làm sạch thông tin thể loại, cờ F2P (`is_free`) và cờ nạp thẻ (`offersIAP`).
3. Quy đổi điểm số đánh giá 0 - 5 sao sang thang điểm chuẩn 0 - 100.
4. Tính doanh thu ước tính kết hợp ARPU thể loại từ dữ liệu vĩ mô.

## Bước 1: Chuẩn bị thư viện và đường dẫn

Nạp các thư viện cần thiết, khai báo đường dẫn tới dữ liệu Mobile, bảng doanh thu theo genre, bảng doanh thu khu vực và các file đầu ra.

In [1]:
import pandas as pd
import numpy as np
import os

# Đường dẫn dữ liệu
RAW_DATA_PATH = os.path.join('..', '..', 'data', 'raw', 'mobile_playstore_games_2000+.csv')
GENRE_REV_PATH = os.path.join('..', '..', 'data', 'external', 'genre_revenue_2024_2025.csv')
REGION_REV_PATH = os.path.join('..', '..', 'data', 'external', 'revenue_by_region(2021 - 2025).csv')
PROCESSED_REGION_PATH = os.path.join('..', '..', 'data', 'processed', 'revenue_by_region_cleaned.csv')
PROCESSED_DATA_PATH = os.path.join('..', '..', 'data', 'processed', 'mobile_cleaned.csv')

print(f"Raw data path: {RAW_DATA_PATH}")

Raw data path: ..\..\data\raw\mobile_playstore_games_2000+.csv


## Bước 2: Đọc và lọc dữ liệu

Đọc dữ liệu Mobile cùng benchmark doanh thu theo genre. Loại game có rating bằng 0 hoặc thiếu năm phát hành để tránh đưa các bản ghi này vào phân tích rating và thời gian.

In [2]:
df_mobile = pd.read_csv(RAW_DATA_PATH)
df_genre_rev = pd.read_csv(GENRE_REV_PATH)

# Loại game có rating bằng 0 hoặc thiếu/năm phát hành bằng 0 trước ETL.
df_mobile["score"] = pd.to_numeric(df_mobile["score"], errors="coerce")
df_mobile["release_year"] = pd.to_numeric(df_mobile["release_year"], errors="coerce")
rows_before_filter = len(df_mobile)
df_mobile = df_mobile.loc[
    df_mobile["score"].gt(0)
    & df_mobile["release_year"].notna()
    & df_mobile["release_year"].gt(0)
].copy()
print(f"Đã loại {rows_before_filter - len(df_mobile)} dòng có rating = 0 hoặc thiếu năm phát hành")
print(f"Số dòng còn lại: {len(df_mobile)}")

df_mobile.head(3)

Đã loại 78 dòng có rating = 0 hoặc thiếu năm phát hành
Số dòng còn lại: 2642


,app_id,title,genre,genre_id,realInstalls,min_installs,installs_range,score,ratings,reviews,...,offersIAP,inAppProductPrice,developer,developer_id,released,release_year,version,content_rating,description,playstore_url
0,com.alexanderwinn.landfall,TerraGenesis: Landfall,Simulation,GAME_SIMULATION,1733498,1000000,"1,000,000+",4.07,26086,2006,...,True,$0.49 - $99.99 per item,Tilting Point,5479878731323183535,"Aug 9, 2022",2022.0,2.92,Everyone,From the makers of TerraGenesis comes TERRAGEN...,https://play.google.com/store/apps/details?id=...
1,com.rsgapps.reactor,Reactor - Energy Sector Tycoon,Strategy,GAME_STRATEGY,9415262,5000000,"5,000,000+",4.44,573229,15419,...,True,$0.99 - $9.99 per item,RSGapps - Idle Tycoon Games,6957790963724025053,"Mar 4, 2016",2016.0,1.72.65,Everyone,Become an industrial idle tycoon by managing y...,https://play.google.com/store/apps/details?id=...
2,com.tankionline.mobile.production,Tanki Online: PvP Tank Battle,Arcade,GAME_ARCADE,4830368,1000000,"1,000,000+",3.35,157853,1612,...,True,$4.99 - $248.99 per item,Alternativa Games,9219768540245481297,"Oct 7, 2018",2018.0,2.6.0.2002432142,Everyone 10+,Dive into the exhilarating world of Tanki Onli...,https://play.google.com/store/apps/details?id=...


## Bước 3: Chuẩn hóa lượt cài, genre và thông tin giá

Dùng realInstalls làm Volume_Units, ánh xạ genre nguồn sang taxonomy benchmark, đồng thời làm sạch các cờ miễn phí, mua hàng trong ứng dụng và giá game.

In [3]:
# Gán realInstalls làm Volume_Units.
df_mobile['Volume_Units'] = df_mobile['realInstalls'].fillna(0).astype(float)

# Chuẩn hóa genre theo taxonomy của bảng benchmark.
genre_mapping = {
    'Role Playing': 'RPG',
    'Board': 'Tabletop',
    'Card': 'Tabletop',
    'Casual': 'Puzzle',
    'Word': 'Puzzle',
    'Trivia': 'Puzzle',
    'Educational': 'Puzzle',
    'Music': 'Arcade',
    'Adventure': 'Action'
}
df_mobile['Genre_Clean'] = df_mobile['genre'].replace(genre_mapping)

# Làm sạch cờ game miễn phí và mua hàng trong ứng dụng.
df_mobile['is_free'] = df_mobile['is_free'].fillna(True).astype(bool)
df_mobile['offersIAP'] = df_mobile['offersIAP'].fillna(False).astype(bool)
df_mobile['price_usd'] = df_mobile['price_usd'].fillna(0.0)

df_mobile[['title', 'Volume_Units', 'Genre_Clean', 'is_free', 'offersIAP']].head()

,title,Volume_Units,Genre_Clean,is_free,offersIAP
0,TerraGenesis: Landfall,1733498.0,Simulation,True,True
1,Reactor - Energy Sector Tycoon,9415262.0,Strategy,True,True
2,Tanki Online: PvP Tank Battle,4830368.0,Arcade,True,True
3,Golf Blitz,3181690.0,Sports,True,True
4,Shadow Fight 3 - RPG fighting,210874943.0,RPG,True,True


## Bước 4: Chuẩn hóa rating và phân hạng

Chuyển điểm Google Play từ thang 0–5 sang thang 0–100, sau đó tạo Rating_Tier theo các ngưỡng đã thống nhất trong kế hoạch ETL.

In [4]:
# Chuyển điểm 0-5 sao sang thang 0-100.
df_mobile['Standardized_Rating'] = (df_mobile['score'] * 20).round(2)

def assign_rating_tier(score):
    if score >= 85:
        return 'Masterpiece'
    elif score >= 75:
        return 'Great'
    elif score >= 50:
        return 'Mixed'
    return 'Poor'

df_mobile['Rating_Tier'] = df_mobile['Standardized_Rating'].apply(assign_rating_tier)
df_mobile[['title', 'score', 'Standardized_Rating', 'Rating_Tier']].head()

,title,score,Standardized_Rating,Rating_Tier
0,TerraGenesis: Landfall,4.07,81.4,Great
1,Reactor - Energy Sector Tycoon,4.44,88.8,Masterpiece
2,Tanki Online: PvP Tank Battle,3.35,67.0,Mixed
3,Golf Blitz,4.27,85.4,Masterpiece
4,Shadow Fight 3 - RPG fighting,4.59,91.8,Masterpiece


## Bước 5: Ước tính ARPU và doanh thu

Tính ARPU theo genre từ doanh thu benchmark năm 2024 và tổng lượt cài trong dữ liệu Mobile. Dùng ARPU cùng lượt cài và giá game để tạo Harmonized_Revenue_USD; đây là giá trị ước tính theo benchmark.

In [5]:
# Tính ARPU theo genre = doanh thu benchmark chia tổng lượt cài.
genre_installs = df_mobile.groupby('Genre_Clean')['realInstalls'].sum()
genre_rev_lookup = df_genre_rev.set_index('Genre')['Mobile_Revenue_2024_USD_Billion'] * 1e9

arpu_dict = (genre_rev_lookup / genre_installs).dropna().to_dict()
df_mobile['ARPU'] = df_mobile['Genre_Clean'].map(arpu_dict)
unmatched = df_mobile.loc[df_mobile['ARPU'].isna(), 'Genre_Clean'].unique()
if len(unmatched) > 0:
    print(f"Genre chưa có ARPU trong benchmark: {unmatched}")

# Ước tính doanh thu chuẩn hóa từ lượt cài, ARPU genre và giá game.
df_mobile['Harmonized_Revenue_USD'] = (df_mobile['realInstalls'] * df_mobile['ARPU']) + (df_mobile['price_usd'] * df_mobile['realInstalls'])

df_mobile[['title', 'realInstalls', 'ARPU', 'Harmonized_Revenue_USD']].head()

,title,realInstalls,ARPU,Harmonized_Revenue_USD
0,TerraGenesis: Landfall,1733498,1.243301,2.155260e+06
1,Reactor - Energy Sector Tycoon,9415262,7.077709,6.663848e+07
2,Tanki Online: PvP Tank Battle,4830368,0.293208,1.416302e+06
3,Golf Blitz,3181690,0.875715,2.786255e+06
4,Shadow Fight 3 - RPG fighting,210874943,8.601021,1.813740e+09


## Bước 6: Tạo và lưu bảng Mobile theo schema master

Chuẩn hóa các trường về schema 12 cột dùng chung. Lưu kết quả vào data/processed/mobile_cleaned.csv để dùng ở bước ghép master.

In [6]:
clean_mobile = pd.DataFrame({
    'Game_Title': df_mobile['title'].astype(str).str.strip(),
    'Platform_Category': 'Mobile',
    'Specific_Platform': 'Android',
    'Genre': df_mobile['Genre_Clean'],
    'Publisher': df_mobile['developer'].fillna('Unknown'),
    'Developer': df_mobile['developer'].fillna('Unknown'),
    'Release_Year': df_mobile['release_year'].astype(int),
    'Volume_Units': df_mobile['Volume_Units'],
    'Harmonized_Revenue_USD': df_mobile['Harmonized_Revenue_USD'],
    'Standardized_Rating': df_mobile['Standardized_Rating'],
    'Rating_Tier': df_mobile['Rating_Tier'],
    'Is_CrossPlatform': False
})

os.makedirs(os.path.dirname(PROCESSED_DATA_PATH), exist_ok=True)
clean_mobile.to_csv(PROCESSED_DATA_PATH, index=False)

print(f"Đã lưu thành công {len(clean_mobile)} dòng vào: {PROCESSED_DATA_PATH}")
clean_mobile.head()

Đã lưu thành công 2642 dòng vào: ..\..\data\processed\mobile_cleaned.csv


,Game_Title,Platform_Category,Specific_Platform,Genre,Publisher,Developer,Release_Year,Volume_Units,Harmonized_Revenue_USD,Standardized_Rating,Rating_Tier,Is_CrossPlatform
0,TerraGenesis: Landfall,Mobile,Android,Simulation,Tilting Point,Tilting Point,2022,1733498.0,2.155260e+06,81.4,Great,False
1,Reactor - Energy Sector Tycoon,Mobile,Android,Strategy,RSGapps - Idle Tycoon Games,RSGapps - Idle Tycoon Games,2016,9415262.0,6.663848e+07,88.8,Masterpiece,False
2,Tanki Online: PvP Tank Battle,Mobile,Android,Arcade,Alternativa Games,Alternativa Games,2018,4830368.0,1.416302e+06,67.0,Mixed,False
3,Golf Blitz,Mobile,Android,Sports,Noodlecake,Noodlecake,2019,3181690.0,2.786255e+06,85.4,Masterpiece,False
4,Shadow Fight 3 - RPG fighting,Mobile,Android,RPG,NEKKI,NEKKI,2017,210874943.0,1.813740e+09,91.8,Masterpiece,False


## Bước 7: Làm sạch bảng doanh thu theo khu vực

Đọc CSV với keep_default_na=False để giữ mã vùng NA, chuẩn hóa kiểu dữ liệu, điền 0 cho tăng trưởng YoY năm cơ sở 2021 và lưu file đã làm sạch.

In [7]:
# 1. Dùng keep_default_na=False để giữ nguyên mã vùng NA.
df_region_raw = pd.read_csv(REGION_REV_PATH, keep_default_na=False)

# 2. Chuẩn hóa kiểu dữ liệu và xử lý giá trị trống.
df_region = df_region_raw.copy()
df_region["Region"] = df_region["Region"].astype(str).str.strip()
df_region["Region_Code"] = df_region["Region_Code"].astype(str).str.strip()
df_region["Year"] = df_region["Year"].astype(int)
df_region["Share_Pct"] = pd.to_numeric(
    df_region["Share_Pct"], errors="coerce"
).fillna(0.0)

# Năm cơ sở 2021 không có tăng trưởng YoY; điền 0.0.
df_region["YoY_Growth_Pct"] = pd.to_numeric(
    df_region["YoY_Growth_Pct"], errors="coerce"
).fillna(0.0)

# 3. Kiểm tra mã vùng NA được giữ nguyên.
print(
    f"Mã vùng NA còn nguyên vẹn: {('NA' in df_region['Region_Code'].values)}"
)
print(f"Số lượng missing value:\n{df_region.isnull().sum()}")

# 4. Lưu bảng đã làm sạch vào thư mục processed.
os.makedirs(os.path.dirname(PROCESSED_REGION_PATH), exist_ok=True)
df_region.to_csv(PROCESSED_REGION_PATH, index=False)

print(f"Đã lưu thành công {len(df_region)} dòng vào: {PROCESSED_REGION_PATH}")
df_region.head(6)

Mã vùng NA còn nguyên vẹn: True
Số lượng missing value:
Region            0
Region_Code       0
Year              0
Share_Pct         0
YoY_Growth_Pct    0
dtype: int64
Đã lưu thành công 25 dòng vào: ..\..\data\processed\revenue_by_region_cleaned.csv


,Region,Region_Code,Year,Share_Pct,YoY_Growth_Pct
0,Asia-Pacific,APAC,2021,48.3,0.0
1,North America,NA,2021,26.5,0.0
2,Europe,EU,2021,17.7,0.0
3,Latin America,LATAM,2021,4.2,0.0
4,Middle East & Africa,MENA,2021,3.3,0.0
5,Asia-Pacific,APAC,2022,47.7,-5.6


## Bước 8: Xuất biểu đồ EDA tĩnh cho Mobile

Tạo năm biểu đồ phân phối, tương quan, lượt cài theo genre, số game theo genre và rating so với doanh thu. Ảnh được lưu trong visuals/eda_static/mobile/; boxplot dùng genre vì hiện chỉ phân tích nền tảng Mobile.

In [8]:
# EDA Mobile: lưu 5 biểu đồ tĩnh vào visuals/eda_static/mobile.
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

project_root = Path.cwd().resolve()
if not (project_root / "data").exists():
    project_root = (project_root / ".." / "..").resolve()
processed_path = project_root / "data" / "processed" / "mobile_cleaned.csv"
output_dir = project_root / "visuals" / "eda_static" / "mobile"
output_dir.mkdir(parents=True, exist_ok=True)

df_eda = pd.read_csv(processed_path)
df_eda["Volume_Units"] = pd.to_numeric(df_eda["Volume_Units"], errors="coerce")
df_eda["Harmonized_Revenue_USD"] = pd.to_numeric(df_eda["Harmonized_Revenue_USD"], errors="coerce")
df_eda["Standardized_Rating"] = pd.to_numeric(df_eda["Standardized_Rating"], errors="coerce")
df_eda["Release_Year"] = pd.to_numeric(df_eda["Release_Year"], errors="coerce")
sns.set_theme(style="whitegrid", context="notebook", palette="Set2")

# 1. Phân phối doanh thu: histogram và KDE trên log10 doanh thu USD.
revenue = df_eda.loc[df_eda["Harmonized_Revenue_USD"] > 0, "Harmonized_Revenue_USD"].dropna()
fig, ax = plt.subplots(figsize=(10, 6))
sns.histplot(np.log10(revenue), bins=35, kde=True, color="#4472C4", ax=ax)
ax.set(title="Mobile: Distribution of Estimated Revenue", xlabel="log10(Harmonized Revenue, USD)", ylabel="Number of games")
fig.tight_layout()
fig.savefig(output_dir / "eda_1_distribution.png", dpi=300, bbox_inches="tight")
plt.close(fig)

# 2. Heatmap tương quan; log1p giúp giảm độ lệch của lượt cài và doanh thu.
corr_data = pd.DataFrame({
    "Release year": df_eda["Release_Year"],
    "log1p(Volume)": np.log1p(df_eda["Volume_Units"].clip(lower=0)),
    "log1p(Revenue)": np.log1p(df_eda["Harmonized_Revenue_USD"].clip(lower=0)),
    "Rating / 100": df_eda["Standardized_Rating"],
}).dropna()
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr_data.corr(), annot=True, fmt=".2f", cmap="vlag", center=0, vmin=-1, vmax=1, square=True, ax=ax)
ax.set_title("Mobile: Correlation Between Numeric Fields")
fig.tight_layout()
fig.savefig(output_dir / "eda_2_correlation.png", dpi=300, bbox_inches="tight")
plt.close(fig)

# 3. Boxplot lượt cài theo genre trong tập Mobile.
box_data = df_eda.loc[(df_eda["Volume_Units"] > 0) & df_eda["Genre"].notna()].copy()
box_data["log10_Volume"] = np.log10(box_data["Volume_Units"])
genre_order = box_data.groupby("Genre")["Volume_Units"].median().sort_values().index
fig, ax = plt.subplots(figsize=(11, 7))
sns.boxplot(data=box_data, x="log10_Volume", y="Genre", order=genre_order, showfliers=False, color="#8FC7B5", ax=ax)
ax.set(title="Mobile: Install Volume Distribution by Genre", xlabel="log10(Volume Units)", ylabel="Genre")
fig.tight_layout()
fig.savefig(output_dir / "eda_3_volume_by_genre.png", dpi=300, bbox_inches="tight")
plt.close(fig)

# 4. Số lượng game theo genre.
fig, ax = plt.subplots(figsize=(11, 6))
order = df_eda["Genre"].value_counts().index
sns.countplot(data=df_eda, y="Genre", order=order, color="#70AD47", ax=ax)
ax.set(title="Mobile: Number of Games by Genre", xlabel="Number of games", ylabel="Genre")
for container in ax.containers:
    ax.bar_label(container, padding=3, fontsize=8)
fig.tight_layout()
fig.savefig(output_dir / "eda_4_genre_count.png", dpi=300, bbox_inches="tight")
plt.close(fig)

# 5. Rating so với doanh thu, trục doanh thu dùng log10.
scatter_data = df_eda.loc[(df_eda["Harmonized_Revenue_USD"] > 0) & df_eda["Standardized_Rating"].notna()].copy()
scatter_data["log10_Revenue"] = np.log10(scatter_data["Harmonized_Revenue_USD"])
fig, ax = plt.subplots(figsize=(10, 6))
sns.scatterplot(data=scatter_data, x="Standardized_Rating", y="log10_Revenue", hue="Genre", alpha=0.65, s=35, palette="tab10", ax=ax)
ax.set(title="Mobile: Rating vs Estimated Revenue", xlabel="Standardized rating (0?100)", ylabel="log10(Harmonized Revenue, USD)")
ax.legend(title="Genre", bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0)
fig.tight_layout()
fig.savefig(output_dir / "eda_5_scatter.png", dpi=300, bbox_inches="tight")
plt.close(fig)

print(f"Saved 5 Mobile EDA charts to: {output_dir}")


Saved 5 Mobile EDA charts to: D:\Study\TTDL\Gaming-Market-Industry-Analytics\visuals\eda_static\mobile
